# Hello World

Get started with BRepAX in 5 minutes.

In [ ]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from brepax.primitives import Disk, Sphere

## 2D: Disk primitive

In [ ]:
disk = Disk(center=jnp.array([0.0, 0.0]), radius=jnp.array(1.0))

# Evaluate SDF at a query point
query = jnp.array([0.5, 0.0])
print(f"SDF at {query}: {disk.sdf(query):.4f}")
print(f"SDF at boundary: {disk.sdf(jnp.array([1.0, 0.0])):.4f}")
print(f"SDF outside: {disk.sdf(jnp.array([2.0, 0.0])):.4f}")

## Gradients flow through everything

In [ ]:
# Gradient of SDF w.r.t. query point
grad_fn = jax.grad(lambda x: disk.sdf(x))
gradient = grad_fn(jnp.array([2.0, 0.0]))
print(f"SDF gradient at (2,0): {gradient}")

# Gradient of SDF w.r.t. radius
import equinox as eqx

grad_disk = eqx.filter_grad(lambda d: d.sdf(jnp.array([2.0, 0.0])))(disk)
print(f"d(SDF)/d(radius): {grad_disk.radius:.4f}")

## 3D: Sphere primitive

In [ ]:
sphere = Sphere(center=jnp.array([0.0, 0.0, 0.0]), radius=jnp.array(1.0))

points = jnp.array(
    [
        [0.0, 0.0, 0.0],  # center (inside)
        [1.0, 0.0, 0.0],  # surface
        [2.0, 0.0, 0.0],  # outside
    ]
)

sdfs = eqx.filter_vmap(sphere.sdf)(points)
print(f"SDF values: {sdfs}")
print(f"Analytical volume: {sphere.volume():.4f}")

## Boolean operations

In [ ]:
from brepax.boolean import union_area

disk_a = Disk(center=jnp.array([0.0, 0.0]), radius=jnp.array(1.0))
disk_b = Disk(center=jnp.array([1.5, 0.0]), radius=jnp.array(1.0))

area = union_area(disk_a, disk_b, method="stratum")
print(f"Union area: {area:.4f}")

# Gradient of union area w.r.t. radius
grad = jax.grad(
    lambda r: union_area(
        Disk(center=jnp.array([0.0, 0.0]), radius=r),
        disk_b,
        method="stratum",
    )
)(jnp.array(1.0))
print(f"d(union_area)/d(r1): {grad:.4f}")